# Benchmark Runner — Planetoid (Cora / CiteSeer / PubMed)

Runs Full Graph, Static PPR (LCILP-style), and Static k-hop on Cora, CiteSeer, PubMed.
Uses real PyG node features — no random embeddings.

| Method | Subgraph | Output |
|--------|----------|--------|
| Full Graph | None | `results/benchmark/{ds}/{enc}/` |
| Static PPR | PPR + sweep cut | `results/benchmark-ppr/{ds}/{enc}_a{a}_e{e}/` |
| Static k-hop | k-hop | `results/benchmark-khop/{ds}/{enc}_k{k}/` |

After running this + `learnable_ppr_planetoid.ipynb`, open `benchmark_analysis.ipynb` to compare.

In [ ]:
SEED = 42
DATASETS  = ['Cora', 'CiteSeer', 'PubMed']
ENCODERS  = ['GCN', 'SAGE', 'GAT']

# Architecture (feature dim is read per-dataset from real node features)
HIDDEN_CHANNELS = 256
NUM_LAYERS      = 3
DROPOUT         = 0.3

# Full-graph training
FULL_EPOCHS     = 500
FULL_BATCH_SIZE = 8192
FULL_LR         = 0.005
FULL_PATIENCE   = 30

# Subgraph-based training (PPR, k-hop)
# Planetoid graphs are small — EDGES_PER_EPOCH=None uses all training edges
SUB_EPOCHS          = 500
SUB_BATCH_SIZE      = 1024
SUB_LR              = 0.005
SUB_PATIENCE        = 30
EDGES_PER_EPOCH     = None      # None = all (Cora ~4.7K, CiteSeer ~3.7K, PubMed ~8K)
EVAL_STEPS          = 5
WEIGHT_DECAY        = 1e-5
GRAD_CLIP           = 1.0

ENCODER_LR_OVERRIDE   = {'GAT': 0.001}
ENCODER_CLIP_OVERRIDE = {'GAT': 0.5}

# Static PPR
PPR_ALPHAS    = [0.85]
PPR_EPSILONS  = [1e-3]
PPR_WINDOW    = 10
PPR_NUM_NEGS  = 5      # number of independent neg caches (rotating global negatives)
PPR_STRUCT_DIM = 2     # (π_u(w), π_v(w)) structural label per node
PPR_EPSILON_PRECOMP = 1e-4  # precision for full per-node PPR precomputation

# Static k-hop
K_VALUES = [2]

import torch
GPU_ID = 0
DEVICE = f'cuda:{GPU_ID}' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}  |  Datasets: {DATASETS}  |  Encoders: {ENCODERS}')

In [ ]:
import numpy as np, random, json, os, time

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from subgrapher.benchmark.data_prep import prepare_planetoid_data
from subgrapher.utils.models import GCN, SAGE, GAT, LinkPredictor


def make_encoder(enc_type, in_ch, hid=HIDDEN_CHANNELS,
                 n_layers=NUM_LAYERS, dropout=DROPOUT):
    if enc_type == 'GCN':  return GCN(in_ch, hid, hid, n_layers, dropout)
    if enc_type == 'SAGE': return SAGE(in_ch, hid, hid, n_layers, dropout)
    if enc_type == 'GAT':  return GAT(in_ch, hid, hid, n_layers, dropout, heads=4)
    raise ValueError(enc_type)

def make_predictor(hid=HIDDEN_CHANNELS, n_layers=NUM_LAYERS, dropout=DROPOUT):
    return LinkPredictor(hid, hid, 1, n_layers, dropout)

def enc_lr(enc_type, base=SUB_LR):
    return ENCODER_LR_OVERRIDE.get(enc_type, base)

def enc_clip(enc_type, base=GRAD_CLIP):
    return ENCODER_CLIP_OVERRIDE.get(enc_type, base)

def ensure_train_only_edges(data, dd):
    if not getattr(data, '_edge_index_train_only', False):
        data._orig_edge_index = data.edge_index
        data.edge_index = dd['train_edge_index']
        data._edge_index_train_only = True
        print(f'  [train-only edges] swapped: '
              f"{data._orig_edge_index.size(1):,} -> {data.edge_index.size(1):,}")
    return data

def save_full_results(base_dir, result_dict):
    run_id = time.strftime('%Y%m%d_%H%M%S')
    result_dict['run_id'] = run_id
    result_dict['timestamp'] = time.strftime('%Y-%m-%d %H:%M:%S')
    run_dir = os.path.join(base_dir, 'runs', run_id)
    os.makedirs(run_dir, exist_ok=True)
    run_path = os.path.join(run_dir, 'full_results.json')
    with open(run_path, 'w') as f:
        json.dump(result_dict, f, indent=2, default=str)
    print(f'  -> {run_path}')
    os.makedirs(base_dir, exist_ok=True)
    latest_path = os.path.join(base_dir, 'full_results.json')
    with open(latest_path, 'w') as f:
        json.dump(result_dict, f, indent=2, default=str)
    print(f'  -> {latest_path} (latest)')

print('Ready.')

## 1. Load Datasets

In [ ]:
datasets = {}
for ds_name in DATASETS:
    print(f'\nLoading {ds_name}...')
    dd = prepare_planetoid_data(ds_name)
    datasets[ds_name] = dd
    data = dd['data']; se = dd['split_edge']
    print(f'  Nodes: {data.num_nodes:,}  Edges: {data.edge_index.size(1):,}  '
          f'Feature dim: {dd["feature_dim"]}')
    print(f'  Train: {se["train"]["source_node"].size(0):,}  '
          f'Val: {se["valid"]["source_node"].size(0):,}  '
          f'Test: {se["test"]["source_node"].size(0):,}')

In [ ]:
for ds_name in DATASETS:
    print(f'[{ds_name}]')
    ensure_train_only_edges(datasets[ds_name]['data'], datasets[ds_name])

In [ ]:
from subgrapher.utils.ppr_preprocessor import PPRPreprocessor

ppr_preprocessors = {}
for ds_name in DATASETS:
    dd = datasets[ds_name]
    ppr_path = f'cache/benchmark-ppr/{ds_name}/ppr_alpha{PPR_ALPHAS[0]}_eps{PPR_EPSILON_PRECOMP}.pt'
    if os.path.isfile(ppr_path):
        print(f'[PPR] Loading preprocessor for {ds_name} from {ppr_path}')
        ppr_preprocessors[ds_name] = PPRPreprocessor.load(ppr_path, dd['data'])
    else:
        print(f'[PPR] Building preprocessor for {ds_name} (one-time cost)...')
        ppr_pre = PPRPreprocessor(dd['data'], ppr_alpha=PPR_ALPHAS[0],
                                  epsilon=PPR_EPSILON_PRECOMP)
        os.makedirs(os.path.dirname(ppr_path), exist_ok=True)
        ppr_pre.save(ppr_path)
        ppr_preprocessors[ds_name] = ppr_pre
    print(f'  {ppr_preprocessors[ds_name]}')

## 2. Full Graph (No Subgraph)

In [ ]:
from subgrapher.benchmark.trainer import benchmark_model

In [ ]:
for ds_name in DATASETS:
    dd = datasets[ds_name]
    data = dd['data']; split_edge = dd['split_edge']
    ensure_train_only_edges(data, dd)
    in_ch = dd['feature_dim']

    for enc_type in ENCODERS:
        print(f'\n=== Full Graph: {ds_name} / {enc_type} ===')
        torch.manual_seed(SEED)
        encoder   = make_encoder(enc_type, in_ch=in_ch)
        predictor = make_predictor()
        encoder.reset_parameters()
        predictor.reset_parameters()

        result = benchmark_model(
            enc_type, encoder, predictor, data, split_edge,
            epochs=FULL_EPOCHS, batch_size=FULL_BATCH_SIZE,
            lr=FULL_LR, eval_steps=EVAL_STEPS, device=DEVICE,
            patience=FULL_PATIENCE, weight_decay=WEIGHT_DECAY,
            grad_clip=GRAD_CLIP)

        save_full_results(
            f'results/benchmark/{ds_name}/{enc_type}',
            {
                'dataset': ds_name,
                'encoder': enc_type,
                'test_results': {k: float(v) for k, v in result['test_results'].items()},
                'train_time': result['train_time'],
                'num_params': result['num_params'],
                'best_epoch': result['best_epoch'],
                'stopped_early': result['stopped_early'],
                'config': {
                    'epochs': FULL_EPOCHS, 'batch_size': FULL_BATCH_SIZE,
                    'lr': FULL_LR, 'patience': FULL_PATIENCE,
                    'feature_dim': in_ch, 'hidden_channels': HIDDEN_CHANNELS,
                    'num_layers': NUM_LAYERS, 'dropout': DROPOUT,
                },
                'seed': SEED,
            })

        del encoder, predictor, result
        torch.cuda.empty_cache()

In [ ]:
import gc
gc.collect(); torch.cuda.empty_cache()
print('Full Graph done.')

## 3. Static PPR Subgraph (LCILP-style)

In [ ]:
from subgrapher.benchmark_ppr.ppr_extractor import StaticPPRExtractor
from subgrapher.benchmark_ppr.trainer_lcilp import train_model_ppr_lcilp
from subgrapher.benchmark_ppr.evaluator import evaluate_ppr_lcilp
from subgrapher.benchmark_ppr.graph_classifier import SubgraphClassifier

In [ ]:
for ds_name in DATASETS:
    dd = datasets[ds_name]
    data = dd['data']; split_edge = dd['split_edge']
    ensure_train_only_edges(data, dd)

    for ppr_alpha in PPR_ALPHAS:
        for ppr_eps in PPR_EPSILONS:
            ppr_ext = StaticPPRExtractor(
                data, alpha=ppr_alpha, epsilon=ppr_eps, window=PPR_WINDOW)

            for enc_type in ENCODERS:
                print(f'\n=== Static PPR LCILP (a={ppr_alpha}, e={ppr_eps}): '
                      f'{ds_name} / {enc_type} ===')
                torch.manual_seed(SEED)
                classifier = SubgraphClassifier(
                    drnl_dim=PPR_STRUCT_DIM, hidden=HIDDEN_CHANNELS,
                    num_layers=NUM_LAYERS, dropout=DROPOUT,
                    encoder_type=enc_type,
                    feature_dim=dd['feature_dim'])
                classifier.reset_parameters()

                cache_dir = f'cache/benchmark-ppr/{ds_name}/a{ppr_alpha}_e{ppr_eps}'
                t0 = time.time()
                hist = train_model_ppr_lcilp(
                    classifier, data, split_edge, ppr_ext,
                    ppr_preprocessor=ppr_preprocessors[ds_name],
                    epochs=SUB_EPOCHS, batch_size=SUB_BATCH_SIZE,
                    lr=enc_lr(enc_type), eval_steps=EVAL_STEPS, device=DEVICE,
                    verbose=True, patience=SUB_PATIENCE,
                    weight_decay=WEIGHT_DECAY, grad_clip=enc_clip(enc_type),
                    num_negs=PPR_NUM_NEGS,
                    edges_per_epoch=EDGES_PER_EPOCH,
                    cache_dir=cache_dir, max_eval_edges=2000)
                train_time = time.time() - t0

                test_res = evaluate_ppr_lcilp(
                    classifier, data, split_edge, ppr_ext,
                    ppr_preprocessor=ppr_preprocessors[ds_name],
                    split='test', batch_size=SUB_BATCH_SIZE, device=DEVICE,
                    max_edges=None, cache_dir=cache_dir)

                save_full_results(
                    f'results/benchmark-ppr/{ds_name}/{enc_type}_a{ppr_alpha}_e{ppr_eps}',
                    {
                        'dataset': ds_name, 'encoder': enc_type,
                        'ppr_alpha': ppr_alpha, 'ppr_epsilon': ppr_eps,
                        'ppr_window': PPR_WINDOW,
                        'test_results': {k: float(v) for k, v in test_res.items()},
                        'train_time': train_time,
                        'best_epoch': hist.get('best_epoch', 0),
                        'stopped_early': hist.get('stopped_early', False),
                        'config': {
                            'ppr_alpha': ppr_alpha, 'ppr_epsilon': ppr_eps,
                            'ppr_window': PPR_WINDOW,
                            'ppr_struct_dim': PPR_STRUCT_DIM, 'num_negs': PPR_NUM_NEGS,
                            'epochs': SUB_EPOCHS, 'batch_size': SUB_BATCH_SIZE,
                            'lr': enc_lr(enc_type), 'patience': SUB_PATIENCE,
                            'hidden_channels': HIDDEN_CHANNELS,
                            'num_layers': NUM_LAYERS, 'dropout': DROPOUT,
                        },
                        'seed': SEED,
                    })

                del classifier, hist, test_res
                torch.cuda.empty_cache()

            del ppr_ext

In [ ]:
import gc
gc.collect(); torch.cuda.empty_cache()
print('Static PPR done.')

## 4. Static k-hop Subgraph

In [ ]:
from subgrapher.benchmark_khop.run_khop_benchmark import load_or_create_khop_preprocessor
from subgrapher.benchmark_khop.khop_extractor import StaticKHopExtractor
from subgrapher.benchmark_khop.trainer_batched import train_model_khop_batched
from subgrapher.benchmark_khop.evaluator import evaluate_khop
from subgrapher.benchmark.evaluator import evaluate_link_prediction

In [ ]:
for ds_name in DATASETS:
    dd = datasets[ds_name]
    data = dd['data']; split_edge = dd['split_edge']
    ensure_train_only_edges(data, dd)
    in_ch = dd['feature_dim']

    for k in K_VALUES:
        khop_pre = load_or_create_khop_preprocessor(ds_name, data, k)
        khop_ext = StaticKHopExtractor(data, k=k, preprocessor=khop_pre)

        for enc_type in ENCODERS:
            print(f'\n=== Static k-hop (k={k}): {ds_name} / {enc_type} ===')
            torch.manual_seed(SEED)
            encoder   = make_encoder(enc_type, in_ch=in_ch)
            predictor = make_predictor()
            encoder.reset_parameters()
            predictor.reset_parameters()

            cache_dir = f'cache/benchmark-khop/{ds_name}/k{k}'
            t0 = time.time()
            hist = train_model_khop_batched(
                encoder, predictor, data, split_edge, khop_ext,
                epochs=SUB_EPOCHS, batch_size=SUB_BATCH_SIZE,
                lr=enc_lr(enc_type), eval_steps=EVAL_STEPS, device=DEVICE,
                verbose=True, patience=SUB_PATIENCE,
                weight_decay=WEIGHT_DECAY, grad_clip=enc_clip(enc_type),
                edges_per_epoch=EDGES_PER_EPOCH,
                cache_dir=cache_dir)
            train_time = time.time() - t0

            test_res = evaluate_khop(
                encoder, predictor, data, split_edge, khop_ext,
                split='test', batch_size=SUB_BATCH_SIZE, device=DEVICE,
                max_edges=None, cache_dir=cache_dir)
            fg_test_res = evaluate_link_prediction(
                encoder, predictor, data, split_edge,
                split='test', batch_size=65536)

            save_full_results(
                f'results/benchmark-khop/{ds_name}/{enc_type}_k{k}',
                {
                    'dataset': ds_name, 'encoder': enc_type, 'k': k,
                    'test_results': {k_: float(v) for k_, v in fg_test_res.items()},
                    'subgraph_test_results': {k_: float(v) for k_, v in test_res.items()},
                    'train_time': train_time,
                    'best_epoch': hist.get('best_epoch', 0),
                    'stopped_early': hist.get('stopped_early', False),
                    'config': {
                        'k': k, 'epochs': SUB_EPOCHS, 'batch_size': SUB_BATCH_SIZE,
                        'lr': enc_lr(enc_type), 'patience': SUB_PATIENCE,
                        'feature_dim': in_ch, 'hidden_channels': HIDDEN_CHANNELS,
                        'num_layers': NUM_LAYERS, 'dropout': DROPOUT,
                    },
                    'seed': SEED,
                })

            del encoder, predictor, hist, test_res, fg_test_res
            torch.cuda.empty_cache()

        del khop_ext, khop_pre

In [ ]:
import gc
gc.collect(); torch.cuda.empty_cache()
print('Static k-hop done.')

## 5. Done

In [ ]:
print('All experiments complete. Results saved to:')
for d in ['results/benchmark', 'results/benchmark-ppr', 'results/benchmark-khop']:
    if os.path.exists(d):
        n = sum(1 for root, _, files in os.walk(d) if 'full_results.json' in files)
        print(f'  {d}: {n} full_results.json files')
print('\nRun learnable_ppr_planetoid.ipynb for Learnable PPR results.')
print('Then open benchmark_analysis.ipynb to compare all methods.')